In [65]:
from pyspark.sql.functions import *

In [66]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("userId", IntegerType(), True),
    StructField("date", StringType(), True),  # can convert to Timestamp later
    StructField("products", ArrayType(
        StructType([
            StructField("productId", IntegerType(), True),
            StructField("quantity", IntegerType(), True)
        ])
    ), True),
    StructField("__v", IntegerType(), True)
])

In [67]:
df = spark.readStream.format('json').schema(schema).load('s3://faker-store-data/raw_carts/')

In [68]:
df= df.withColumn('products',explode("products")
).select(
    "id",
    "userId",
    "date",
    col("products.productId").alias("productId"),
    col("products.quantity").alias("quantity")
)

In [69]:
df.writeStream \
  .format("parquet") \
  .outputMode("append") \
  .trigger(once=True) \
  .option("checkpointLocation", "s3://faker-store-data/delta_carts/checkpoint") \
  .option("path", "s3://faker-store-data/delta_carts/data") \
  .start()

In [70]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, DateType

schema = StructType([
    StructField("category", StringType(), True),
    StructField("description", StringType(), True),
    StructField("id", LongType(), True),
    StructField("image", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("rating", StructType([
        StructField("count", LongType(), True),
        StructField("rate", DoubleType(), True)
    ]), True),
    StructField("title", StringType(), True),
    StructField("date", DateType(), True)
])

In [71]:
df = spark.readStream.format('json').schema(schema).load('s3://faker-store-data/raw_products/')

In [72]:
df = df.select('category','description','id',
       'image','price','rating.count','rating.rate','title')


In [73]:
df.writeStream \
  .format("parquet") \
  .outputMode("append") \
  .trigger(once=True) \
  .option("checkpointLocation", "s3://faker-store-data/delta_products/checkpoint") \
  .option("path", "s3://faker-store-data/delta_products/data") \
  .start()

In [74]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, DateType
)

schema = StructType([
    StructField("__v", LongType(), True),

    StructField("address", StructType([
        StructField("city", StringType(), True),

        StructField("geolocation", StructType([
            StructField("lat", StringType(), True),
            StructField("long", StringType(), True)
        ]), True),

        StructField("number", LongType(), True),
        StructField("street", StringType(), True),
        StructField("zipcode", StringType(), True)
    ]), True),

    StructField("email", StringType(), True),
    StructField("id", LongType(), True),

    StructField("name", StructType([
        StructField("firstname", StringType(), True),
        StructField("lastname", StringType(), True)
    ]), True),

    StructField("password", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("username", StringType(), True),
    StructField("date", DateType(), True)
])

In [75]:
df = spark.readStream.format('json').schema(schema).load('s3://faker-store-data/raw_users/')


In [76]:
df= df.select(
    "id","email","username","password","phone","date",
    col("name.firstname").alias("firstname"),
    col("name.lastname").alias("lastname"),
    col("address.city").alias("city"),
    col("address.street").alias("street"),
    col("address.number").alias("street_number"),
    col("address.zipcode").alias("zipcode"),
    col("address.geolocation.lat").cast("double").alias("lat"),
    col("address.geolocation.long").cast("double").alias("long")
)


In [77]:
df.writeStream \
  .format("parquet") \
  .outputMode("append") \
  .trigger(once=True) \
  .option("checkpointLocation", "s3://faker-store-data/delta_users/checkpoint") \
  .option("path", "s3://faker-store-data/delta_users/data") \
  .start()